# Structured Streaming en Spark

El streaming en ingeniería de datos se refiere al procesamiento continuo y en tiempo real de datos que llegan en forma de flujos. A diferencia del procesamiento por lotes, donde los datos se procesan en bloques, el streaming permite la ingestión y transformación de datos de manera incremental y con baja latencia.

## Características del Streaming

- **Procesamiento Continuo**: Los datos se procesan tan pronto como llegan, lo que permite respuestas rápidas y actualizaciones en tiempo real.
- **Baja Latencia**: Ideal para aplicaciones que requieren resultados inmediatos, como dashboards en vivo y sistemas de alerta.
- **Ingestión Incremental**: Solo se procesan los nuevos datos que llegan desde la última ejecución, optimizando el uso de recursos.

## Fuentes Comunes de Datos de Streaming

- **Apache Kafka**: Un sistema de mensajería distribuido que permite la publicación y suscripción de flujos de datos.
- **Amazon Kinesis**: Un servicio de AWS para la ingestión y procesamiento de datos en tiempo real.
- **Sockets TCP**: Permiten la transmisión de datos en tiempo real a través de redes.

## Uso en Databricks

En Databricks, el streaming se puede implementar utilizando tablas de streaming, que son tablas registradas en Unity Catalog con soporte adicional para el procesamiento incremental de datos. Estas tablas permiten la carga de datos desde fuentes como Kafka y almacenamiento en la nube, y se actualizan automáticamente en cada ejecución.

### Ventajas de las Tablas de Streaming

- **Manejo de Grandes Volúmenes de Datos**: Ideal para flujos de datos que crecen rápidamente.
- **Transformaciones de Baja Latencia**: Permiten razonamientos sobre filas y ventanas de tiempo con alta eficiencia.
- **Actualizaciones Incrementales**: Cada actualización de la tabla de streaming lee la información cambiada en la fuente de streaming y agrega nueva información a la tabla.

El streaming es una técnica poderosa en ingeniería de datos que permite la creación de sistemas reactivos y en tiempo real, optimizando el procesamiento y la respuesta a eventos de datos continuos.


# ¿Qué es el Structured Streaming?

Structured Streaming es un motor de procesamiento en tiempo casi real que ofrece tolerancia a fallos de extremo a extremo con garantías de procesamiento exactamente una vez utilizando las API familiares de Spark. Permite expresar cálculos en datos de streaming de la misma manera que se expresan en datos estáticos por lotes. El motor de Structured Streaming realiza el cálculo de manera incremental y actualiza continuamente el resultado a medida que llegan los datos de streaming.

## Características Principales

- **Procesamiento Incremental**: Los cálculos se realizan de manera incremental y continua.
- **Baja Latencia**: Puede lograr latencias de extremo a extremo tan bajas como 1 milisegundo en modo de procesamiento continuo.
- **Tolerancia a Fallos**: Garantiza exactamente una vez la tolerancia a fallos a través de checkpointing y Write-Ahead Logs.
- **API Familiar**: Utiliza las mismas API de DataFrame/Dataset que se usan para el procesamiento por lotes.

# **Sintaxis**

### **Read Stream**

In [0]:
df_stream = spark.readStream \
  .format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", "my_path/checkpoints/autoloader_schema") \
  .load("my_path/raw_data/")

### **Write Stream**

In [0]:
stream_query = df_transformado.writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("checkpointLocation", "my_path/checkpoints/delta_sink") \
  .trigger(processingTime='1 minute') \
  .toTable("nombre_esquema.tabla_destino")

## Modos de Salida | Output Modes

- **Complete Mode**: Se escribe toda la tabla de resultados actualizada en el almacenamiento externo.
- **Append Mode**: Solo se escriben las nuevas filas añadidas a la tabla de resultados desde el último disparador.
- **Update Mode**: Solo se escriben las filas que se actualizaron en la tabla de resultados desde el último disparador.

Structured Streaming facilita la creación de aplicaciones de procesamiento de datos en tiempo real de manera eficiente y con garantías de consistencia.

In [0]:
# Append Mode
query = df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

# Complete Mode
query = df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

# Update Mode
query = df.writeStream \
    .outputMode("update") \
    .format("console") \
    .start()


# Triggers

La función `.trigger` en Spark Structured Streaming se utiliza para definir el momento en que se procesarán los datos de streaming. Especifica cómo y cuándo se deben generar los micro-lotes o si se debe utilizar el modo de procesamiento continuo. Aquí hay algunos modos comunes de disparadores (triggers):

- **Default trigger**: Ejecuta micro-lotes tan pronto como sea posible. Espera a que termine un lote para empezar con otro.

- **ProcessingTime trigger**: Ejecuta micro-lotes a intervalos de tiempo fijos.

- **Available-now trigger**: Procesa todos los datos disponibles en múltiples micro-lotes y luego se detiene.

- **Continuous trigger**: Ejecuta la consulta en modo de procesamiento continuo con baja latencia.

- **Once trigger**: Ejecuta la consulta una sola vez y procesa todos los datos disponibles en ese momento.

## Ejemplos de Uso

In [0]:
# Default Trigger
df.writeStream \
  .format("console") \
  .start()


# ProcessingTime trigger con intervalo de dos segundos
df.writeStream \
  .format("console") \
  .trigger(processingTime='2 seconds') \
  .start()


# Available-now trigger
df.writeStream \
  .format("console") \
  .trigger(availableNow=True) \
  .start()


# Continuous trigger con intervalo de checkpoint de un segundo
df.writeStream \
  .format("console") \
  .trigger(continuous='1 second') \
  .start()


# Once trigger
df.writeStream \
  .format("console") \
  .trigger(once=True) \
  .start()

# Carpetas

### Input / Source (opcional, depende de la fuente)

- Si lees de un directorio de archivos (ej: CSV, JSON, Parquet en HDFS, S3, local), ese directorio es tu **input path**.

- Spark va a estar mirando ese folder y cada archivo nuevo que caiga lo va a procesar.

- Si la fuente es Kafka, Kinesis, etc., **no necesitás crear carpetas de input**, porque el stream viene directo del sistema.

### Checkpoint Location (OBLIGATORIO en la mayoría de casos)

Es un directorio donde Spark guarda metadatos del stream:

- Offset de cada fuente (por dónde va leyendo)

- Estado de agregaciones con `withWatermark` o `groupBy`

- Commits de micro-batch ya procesados

Sirve para:

- **Recuperación:** si el job se cae, Spark arranca desde el último estado consistente.

- **Exactly-once:** evitar procesar dos veces el mismo batch.

Sin esto, si el job se reinicia, Spark no sabe qué ya procesó → ***duplicados***.

**Ejemplo:**

In [0]:
query = df.writeStream \
    .format("parquet") \
    .option("checkpointLocation", "s3://mi-bucket/checkpoints/ventas/") \
    .option("path", "s3://mi-bucket/output/ventas/") \
    .start()

### Output / Sink

- Es el directorio donde vas a persistir los resultados del stream si escribís a archivos.

- Puede ser parquet, delta, csv, etc.

- **Ojo:** en streaming Spark no sobreescribe, siempre appendea.

### Temp / Metadata interna (opcional)

- Algunas veces Spark o el formato (por ejemplo Delta Lake) crean carpetas auxiliares:

- `_spark_metadata`: info sobre qué archivos ya se procesaron.

- `_delta_log`: si usás Delta Lake, guarda el transaction log.

Estas se crean automáticamente, no hace falta armarlas vos.

### Windowing

En streaming, los datos son unbounded (no tienen fin).
Por lo tanto, no podés ejecutar agregaciones globales como en batch, porque el stream nunca termina:

Windowing es una estrategia de time-based aggregation que transforma un stream infinito en subconjuntos finitos basados en una columna temporal (event-time).

- **Event Time** → cuándo ocurrió el evento (columna del dataset).

- **Processing Time** → cuándo Spark procesa el evento.

### Tipos de Ventana en Spark

#### **Tumbling Window**

Ventanas fijas, no superpuestas.

**Ejemplo:** 5 minutos.

In [0]:
from pyspark.sql.functions import window, col

df.groupBy(
    window(col("event_time"), "5 minutes")
).count()

Si llegan eventos:

10:01

10:02

10:06

10:07

Las ventanas serán:

- [10:00–10:05)

- [10:05–10:10)

#### Sliding Window

Ventanas superpuestas.

**Ejemplo:** Ventana de 10 minutos que se desliza cada 5 minutos.

In [0]:
df.groupBy(
    window(col("event_time"), "10 minutes", "5 minutes")
).count()

Significa:

- Cada ventana dura 10 min
- Se crea una nueva cada 5 min

[10:00–10:10)

[10:05–10:15)

[10:10–10:20)

### Session Window

Basada en inactividad

La ventana se mantiene abierta mientras no haya un gap mayor a 10 minutos.

In [0]:
from pyspark.sql.functions import session_window

df.groupBy(
    session_window(col("event_time"), "10 minutes"),
    col("user_id")
).count()